In [1]:
import numpy as np
import pandas as pd

# 0. Load raw KPI dataset

data_path = "../data/processed/compustat_kpis.csv"
df = pd.read_csv(data_path)

print("Original shape:", df.shape)
display(df.head())

# 1. Keep only IDs + KPI columns

id_cols = ["gvkey", "datadate", "fyear", "conm", "tic"]
id_cols = [c for c in id_cols if c in df.columns]

kpi_cols = [
    "roa",
    "ebitda_margin",
    "debt_ratio",
    "total_debt_to_equity",
    "current_ratio",
    "cash_ratio",
    "interest_coverage",
    "cfo_margin",
    "free_cash_flow",
    "asset_turnover",
    "sales_growth",
    "asset_growth",
    "book_to_market",
    "price_to_book",
    "working_capital_to_assets",
    "retained_earnings_to_assets",
]

kpi_cols = [c for c in kpi_cols if c in df.columns]

print("KPI columns kept:", kpi_cols)
print("Number of KPIs:", len(kpi_cols))

df = df[id_cols + kpi_cols]

print("Shape after column selection:", df.shape)
display(df.head())

# 2. Basic cleaning

df[kpi_cols] = df[kpi_cols].replace([np.inf, -np.inf], np.nan)

df = df.dropna(subset=kpi_cols, how="all")
print("Shape after dropping rows with all KPI = NaN:", df.shape)

row_missing_ratio = df[kpi_cols].isna().mean(axis=1)
df = df[row_missing_ratio <= 0.5]
print("Shape after dropping rows with >50% missing KPIs:", df.shape)

# 3. Fill missing values (median)

for col in kpi_cols:
    median_value = df[col].median(skipna=True)
    df[col] = df[col].fillna(median_value)

print("Number of remaining NaN in KPI columns:", df[kpi_cols].isna().sum().sum())

# 4. Winsorization (1% / 99%)

for col in kpi_cols:
    lower = df[col].quantile(0.01)
    upper = df[col].quantile(0.99)
    df[col] = df[col].clip(lower=lower, upper=upper)

print("Winsorization done (1st and 99th percentiles).")

# 5. Save "clean" version (no scaling)

output_clean = "../data/processed/compustat_kpis_clean.csv"
df.to_csv(output_clean, index=False)
print("Clean file saved to:", output_clean)

# 6. Standardize KPIs (z-score)

df_scaled = df.copy()

for col in kpi_cols:
    mean_val = df_scaled[col].mean()
    std_val = df_scaled[col].std()

    if std_val == 0 or np.isnan(std_val):
        df_scaled[col] = 0.0
    else:
        df_scaled[col] = (df_scaled[col] - mean_val) / std_val

print("Standardization (z-score) done.")

# 7. Save scaled version + summary stats

output_scaled = "../data/processed/compustat_kpis_clean_scaled.csv"
df_scaled.to_csv(output_scaled, index=False)
print("Scaled file saved to:", output_scaled)

summary = df[kpi_cols].describe().T
summary_path = "../data/processed/kpi_summary_stats.csv"
summary.to_csv(summary_path)
print("Summary stats saved to:", summary_path)

print("\nFinal shapes:")
print("Clean (non-scaled):", df.shape)
print("Clean + scaled:", df_scaled.shape)
display(summary.head())


Original shape: (332675, 55)


,gvkey,datadate,fyear,conm,tic,act,lct,at,lt,seq,...,interest_coverage,cfo_margin,free_cash_flow,asset_turnover,sales_growth,asset_growth,working_capital_to_assets,retained_earnings_to_assets,book_to_market,price_to_book
0,1004,1995-05-31,1994,AAR CORP,AIR,321.632,73.140,425.814,228.695,197.119,...,2.242018,0.033795,6.182,1.060076,NaN,NaN,0.583569,0.240565,NaN,1.234818
1,1004,1996-05-31,1995,AAR CORP,AIR,338.012,79.385,437.846,233.211,204.635,...,3.055953,0.049031,17.213,1.153351,0.118732,0.028256,0.590680,0.250182,NaN,1.729691
2,1004,1997-05-31,1996,AAR CORP,AIR,414.100,99.981,529.584,260.325,269.259,...,3.976451,0.016173,-20.761,1.112813,0.167009,0.209521,0.593143,0.231646,NaN,2.095840
3,1004,1998-05-31,1997,AAR CORP,AIR,468.400,149.148,670.559,369.709,300.850,...,4.465020,0.029181,5.328,1.166375,0.327144,0.266200,0.476098,0.220100,NaN,2.434527
4,1004,1999-05-31,1998,AAR CORP,AIR,508.186,173.586,726.630,400.595,326.035,...,4.167663,0.031072,-7.606,1.263416,0.173774,0.083618,0.460482,0.245524,0.602903,1.658646


KPI columns kept: ['roa', 'ebitda_margin', 'debt_ratio', 'total_debt_to_equity', 'current_ratio', 'cash_ratio', 'interest_coverage', 'cfo_margin', 'free_cash_flow', 'asset_turnover', 'sales_growth', 'asset_growth', 'book_to_market', 'price_to_book', 'working_capital_to_assets', 'retained_earnings_to_assets']
Number of KPIs: 16
Shape after column selection: (332675, 21)


,gvkey,datadate,fyear,conm,tic,roa,ebitda_margin,debt_ratio,total_debt_to_equity,current_ratio,...,interest_coverage,cfo_margin,free_cash_flow,asset_turnover,sales_growth,asset_growth,book_to_market,price_to_book,working_capital_to_assets,retained_earnings_to_assets
0,1004,1995-05-31,1994,AAR CORP,AIR,0.031793,0.077019,0.537077,0.615861,4.397484,...,2.242018,0.033795,6.182,1.060076,NaN,NaN,NaN,1.234818,0.583569,0.240565
1,1004,1996-05-31,1995,AAR CORP,AIR,0.049849,0.084273,0.532632,0.585266,4.257882,...,3.055953,0.049031,17.213,1.153351,0.118732,0.028256,NaN,1.729691,0.590680,0.250182
2,1004,1997-05-31,1996,AAR CORP,AIR,0.060621,0.093627,0.491565,0.439324,4.141787,...,3.976451,0.016173,-20.761,1.112813,0.167009,0.209521,NaN,2.095840,0.593143,0.231646
3,1004,1998-05-31,1997,AAR CORP,AIR,0.074896,0.101006,0.551344,0.590813,3.140505,...,4.465020,0.029181,5.328,1.166375,0.327144,0.266200,NaN,2.434527,0.476098,0.220100
4,1004,1999-05-31,1998,AAR CORP,AIR,0.080941,0.102876,0.551305,0.556256,2.927575,...,4.167663,0.031072,-7.606,1.263416,0.173774,0.083618,0.602903,1.658646,0.460482,0.245524


Shape after dropping rows with all KPI = NaN: (276316, 21)
Shape after dropping rows with >50% missing KPIs: (240308, 21)
Number of remaining NaN in KPI columns: 0
Winsorization done (1st and 99th percentiles).
Clean file saved to: ../data/processed/compustat_kpis_clean.csv
Standardization (z-score) done.
Scaled file saved to: ../data/processed/compustat_kpis_clean_scaled.csv
Summary stats saved to: ../data/processed/kpi_summary_stats.csv

Final shapes:
Clean (non-scaled): (240308, 21)
Clean + scaled: (240308, 21)


,count,mean,std,min,25%,50%,75%,max
roa,240308.0,-0.294836,1.444858,-11.815924,-0.071920,0.017381,0.061630,0.348705
ebitda_margin,240308.0,-1.356236,8.377378,-71.087312,0.015394,0.113829,0.235843,0.784319
debt_ratio,240308.0,0.918641,2.100383,0.028815,0.351549,0.590874,0.848984,18.431481
total_debt_to_equity,240308.0,0.791244,2.873902,-11.025820,0.001957,0.335757,1.057264,17.362048
current_ratio,240308.0,2.672375,3.470963,0.012904,1.156315,1.719343,2.632603,23.958693
